In [10]:
import pandas as pd
df = pd.read_parquet("data/block1_train.parquet")
#df2 = pd.read_parquet("data/block2_test.parquet")


In [11]:
#Extra fields
from datetime import datetime


df = df.drop(['postal_code', 'province', 'municipality', 'usage', 'second_driver_birthdate', 'second_driver_claim_free_years', 'vehicle_is_taxi', 'postal_code_houses_owned_by_rental_association_ratio', 'vehicle_number_plate'], axis=1)
#Convert to binary
df['is_driver_owner'] = df['is_driver_owner'].map({'no': 0, 'yes': 1})
df['vehicle_is_marked_for_export'] = df['vehicle_is_marked_for_export'].map({'no': 0, 'yes': 1})
df['vehicle_has_open_recall'] = df['vehicle_has_open_recall'].map({'no': 0, 'yes': 1})
df['vehicle_is_imported_within_last_12_months'] = df['vehicle_is_imported_within_last_12_months'].map({'no': 0, 'yes': 1})
df['vehicle_is_imported'] = df['vehicle_is_imported'].map({'no': 0, 'yes': 1})
df['vehicle_can_be_registered'] = df['vehicle_can_be_registered'].map({'no': 0, 'yes': 1})


categorical = ['coverage', 'payment_frequency', 'is_driver_owner', 'vehicle_is_marked_for_export', 'vehicle_has_open_recall', 'vehicle_is_imported_within_last_12_months', 'vehicle_is_imported', 'vehicle_can_be_registered', 'vehicle_maker', 'vehicle_fuel_type', 'vehicle_primary_color']
df_categorical = df[categorical]
df = df.drop(categorical, axis=1)

date_related_columns = ['contractor_birthdate', 'vehicle_first_registration_date', 'vehicle_country_first_registration_date', 'vehicle_last_registration_date', 'vehicle_inspection_report_date', 'vehicle_inspection_expiry_date']



df_dates = df[date_related_columns]
def days_since_today_for_all_dates(df):
    """
    Converts all columns in the DataFrame to datetime and calculates 
    the number of days since today for each date column.
    
    Parameters:
    - df: pd.DataFrame with only date columns.
    
    Returns:
    - pd.DataFrame with the same columns but values replaced by days since today.
    """
    today = pd.Timestamp(datetime.now().date())
    
    # Convert all columns to datetime and calculate days since today
    df_days = df.apply(lambda col: (today - pd.to_datetime(col, dayfirst=True)).dt.days)
    
    return df_days


df = df.drop(date_related_columns, axis=1)
df_dates = days_since_today_for_all_dates(df_dates)

for c in df.columns:
    df[c] = pd.to_numeric(
        df[c], errors='coerce'
    )


df= pd.concat([df, df_dates,df_categorical], axis=1)

In [12]:
# Find columns where all values are NaN
all_nan_cols = df.columns[df.isna().all()]
print("Columns with all NaN values:")
print(all_nan_cols.tolist())

Columns with all NaN values:
[]


In [7]:
# Extract data into 4 separate DataFrames based on column prefixes

# 1. Columns beginning with postal_code, municipality, or province
df_location = df[[col for col in df.columns if col.startswith(('postal_code', 'municipality', 'province'))]]

# 2. Columns beginning with vehicle
df_vehicle = df[[col for col in df.columns if col.startswith('vehicle')]]

# 3. Columns beginning with Insurer
df_insurer = df[[col for col in df.columns if col.startswith('Insurer') or col.startswith('coverage')]]

# 4. Everything else
df_driver = df[[col for col in df.columns if not any(col.startswith(prefix) for prefix in ['postal_code', 'municipality', 'province', 'vehicle', 'Insurer', 'coverage'])]]

# Move 'vehicle_planned_annual_mileage' from df_vehicle to df_driver
df_driver['vehicle_planned_annual_mileage'] = df_vehicle['vehicle_planned_annual_mileage']
df_vehicle = df_vehicle.drop('vehicle_planned_annual_mileage', axis=1)

# dividing vehicle into 2 dfs - physical characteristics and administrative characteristics
physical_cols = [col for col in df_vehicle.columns if col.startswith((
    'vehicle_make', 'vehicle_model', 'vehicle_year',
    'vehicle_engine_size', 'vehicle_fuel_type', 'vehicle_engine_size',
    'vehicle_powe', 'vehicle_net_weight', 'vehicle_gross_weight',
    'vehicle_length', 'vehicle_width', 'vehicle_height',
    'vehicle_number_of_cylinders', 'vehicle_number_of_doors',
    'vehicle_number_of_seats', 'vehicle_number_of_wheels',
    'vehicle_primary_color', 'vehicle_value_new', 'vehicle_net_max_power',
    'vehicle_net_max_power_electric',
    'vehicle_nominal_continuous_max_power',
    'vehicle_power_to_net_weight_ratio'))]

df_vehicle_physical = df_vehicle[physical_cols]
administrative_cols = [col for col in df_vehicle.columns if col not in physical_cols]
df_vehicle_administrative = df_vehicle[administrative_cols]

In [ ]:
df_vehicle_administrative['vehicle_is_marked_for_export']

vehicle_is_marked_for_export
0    539894
1      1398
Name: count, dtype: int64